In [1]:
# Step 1: Imports
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# Step 2: Load Dataset
data = load_digits()
x = data.images
y_class = data.target

# Step 3: Create Dummy Bounding Boxes (since dataset has none)
# Format: [x1, y1, x2, y2] normalized (0 to 1)
y_bbox = np.array([[0.2, 0.2, 0.8, 0.8]] * len(x))

# Step 4: Preprocess
x = x.reshape(-1, 8, 8, 1) / 16.0

# Combine labels (class + bbox)
y = np.concatenate([y_class.reshape(-1,1), y_bbox], axis=1)

# Step 5: Split
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# Split labels
y_train_class = y_train[:,0]
y_train_bbox  = y_train[:,1:]

y_test_class = y_test[:,0]
y_test_bbox  = y_test[:,1:]

# Step 6: CNN Model (Two Outputs)
input_layer = layers.Input(shape=(8,8,1))

x_layer = layers.Conv2D(32, (3,3), activation='relu')(input_layer)
x_layer = layers.Flatten()(x_layer)
x_layer = layers.Dense(64, activation='relu')(x_layer)

# Output 1: Class
class_output = layers.Dense(10, activation='softmax', name='class')(x_layer)

# Output 2: Bounding Box
bbox_output = layers.Dense(4, activation='sigmoid', name='bbox')(x_layer)

model = models.Model(inputs=input_layer, outputs=[class_output, bbox_output])

# Step 7: Compile
model.compile(
    optimizer='adam',
    loss={
        'class': 'sparse_categorical_crossentropy',
        'bbox': 'mse'
    },
    metrics={'class': 'accuracy'}
)

# Step 8: Train
model.fit(
    x_train,
    {'class': y_train_class, 'bbox': y_train_bbox},
    epochs=5,
    batch_size=32
)

# Step 9: Evaluate
pred_class, pred_bbox = model.predict(x_test)

pred_class = np.argmax(pred_class, axis=1)

print("\nClassification Report:\n")
print(classification_report(y_test_class, pred_class))

# Step 10: IoU Function
def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2-x1) * max(0, y2-y1)

    area1 = (box1[2]-box1[0])*(box1[3]-box1[1])
    area2 = (box2[2]-box2[0])*(box2[3]-box2[1])

    union = area1 + area2 - inter

    return inter / union

# Step 11: Compute Average IoU
ious = []
for i in range(len(pred_bbox)):
    ious.append(iou(y_test_bbox[i], pred_bbox[i]))

print("\nAverage IoU:", np.mean(ious))

Epoch 1/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - bbox_loss: 0.0346 - class_accuracy: 0.2109 - class_loss: 1.6084 - loss: 1.6448
Epoch 2/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - bbox_loss: 0.0103 - class_accuracy: 0.0988 - class_loss: 0.4469 - loss: 0.4575
Epoch 3/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - bbox_loss: 0.0068 - class_accuracy: 0.1009 - class_loss: 0.2331 - loss: 0.2401
Epoch 4/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - bbox_loss: 0.0046 - class_accuracy: 0.0995 - class_loss: 0.1875 - loss: 0.1919
Epoch 5/5
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - bbox_loss: 0.0034 - class_accuracy: 0.1016 - class_loss: 0.1435 - loss: 0.1471
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step

Classification Report:

              precision    recall  f1-score   support

         0.0       1.00      1.00      1.00        33
         1.0       0.96      0.96      0.96        28
         2.0       0.97      0.97      0.97        33
         3.0       1.00      0.97      0.99        34
         4.0   